# Nonlinear Option Pricing - Homework
## Pricing d'une Option Put Américaine avec Réseaux de Neurones

Ce notebook présente la solution pour le pricing d'une option Put Américaine. Nous allons comparer trois approches :
1. **Black-Scholes (Européen)** : Sert de borne inférieure.
2. **Longstaff-Schwartz (Régression Polynomiale)** : La méthode standard pour les options américaines.
3. **Longstaff-Schwartz (Réseaux de Neurones)** : Une approche moderne utilisant le Deep Learning pour estimer la valeur de continuation.

### Paramètres du problème
* $S_0 = 100$
* $K = 95$
* $T = 0.5$ (6 mois)
* $r = 5\%$
* $q = 2\%$
* $\sigma = 20\%$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import torch
import torch.nn as nn
import torch.optim as optim

# Paramètres du modèle
S0 = 100.0      # Prix initial
K = 95.0        # Strike
T = 0.5         # Maturité (années)
r = 0.05        # Taux sans risque
q = 0.02        # Dividende
sigma = 0.2     # Volatilité
M = 50          # Nombre de pas de temps
N = 10000       # Nombre de simulations
dt = T / M      # Pas de temps

In [ ]:
# 1. Prix Black-Scholes (Borne Inférieure Européenne)
def black_scholes_put(S, K, T, r, q, sigma):
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    put = K * np.exp(-r * T) * norm.cdf(-d2) - S * np.exp(-q * T) * norm.cdf(-d1)
    return put

bs_price = black_scholes_put(S0, K, T, r, q, sigma)
print(f"Prix Put Européen (Black-Scholes) : {bs_price:.4f}")

In [ ]:
# 2. Simulation Monte Carlo des trajectoires
np.random.seed(42)
S = np.zeros((N, M + 1))
S[:, 0] = S0
for t in range(1, M + 1):
    z = np.random.standard_normal(N)
    S[:, t] = S[:, t - 1] * np.exp((r - q - 0.5 * sigma ** 2) * dt + sigma * np.sqrt(dt) * z)

# Fonction de Payoff (Put)
def payoff(S, K):
    return np.maximum(K - S, 0)

# Visualisation de quelques trajectoires
plt.figure(figsize=(10, 6))
plt.plot(np.linspace(0, T, M + 1), S[:10, :].T)
plt.title("Trajectoires simulées (10 premiers chemins)")
plt.xlabel("Temps (années)")
plt.ylabel("Prix de l'actif")
plt.grid(True)
plt.show()

## Méthode de Longstaff-Schwartz (Régression Polynomiale)

Cette méthode utilise une régression des moindres carrés pour estimer l'espérance conditionnelle de la valeur de continuation. Nous utilisons ici un polynôme de degré 2.

In [ ]:
# Matrice des flux de trésorerie
cashflows = payoff(S, K)
# Matrice de valeur (initialisée à la maturité)
value_matrix = np.zeros_like(S)
value_matrix[:, -1] = cashflows[:, -1]

discount_factor = np.exp(-r * dt)

# Algorithme Backward
for t in range(M - 1, 0, -1):
    # On considère uniquement les chemins dans la monnaie (ITM)
    itm_indices = np.where(payoff(S[:, t], K) > 0)[0]
    
    if len(itm_indices) > 0:
        X = S[itm_indices, t]
        # Valeur actualisée du pas suivant
        Y = value_matrix[itm_indices, t + 1] * discount_factor
        
        # Régression polynomiale (degré 2)
        coeffs = np.polyfit(X, Y, 2)
        continuation_value = np.polyval(coeffs, X)
        
        exercise_value = payoff(X, K)
        
        # Décision d'exercice : Exercice immédiat > Valeur de continuation
        exercise = exercise_value > continuation_value
        
        # Mise à jour de la matrice de valeur
        # Si exercice, on prend la valeur d'exercice, sinon on actualise la valeur future
        value_matrix[itm_indices, t] = np.where(exercise, exercise_value, Y)
        
    # Pour les chemins hors de la monnaie (OTM), on actualise simplement
    otm_indices = np.where(payoff(S[:, t], K) == 0)[0]
    value_matrix[otm_indices, t] = value_matrix[otm_indices, t + 1] * discount_factor

ls_poly_price = np.mean(value_matrix[:, 1] * discount_factor)
print(f"Prix Put Américain (Longstaff-Schwartz Poly) : {ls_poly_price:.4f}")

## Méthode de Longstaff-Schwartz avec Réseaux de Neurones (Deep Optimal Stopping)

Au lieu d'une régression polynomiale, nous utilisons un réseau de neurones pour apprendre la fonction de valeur de continuation à chaque pas de temps. Cette approche est particulièrement utile en grande dimension, bien que nous l'illustrions ici en dimension 1.

In [ ]:
# Réinitialisation de la matrice de valeur
value_matrix_nn = np.zeros_like(S)
value_matrix_nn[:, -1] = cashflows[:, -1]

# Définition du Réseau de Neurones
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(1, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Paramètres d'entraînement
epochs = 50
learning_rate = 0.01

print("Début de l'entraînement du réseau de neurones (Backward)...")

for t in range(M - 1, 0, -1):
    itm_indices = np.where(payoff(S[:, t], K) > 0)[0]
    
    if len(itm_indices) > 0:
        # Préparation des données pour PyTorch
        X_np = S[itm_indices, t].reshape(-1, 1).astype(np.float32)
        Y_np = (value_matrix_nn[itm_indices, t + 1] * discount_factor).reshape(-1, 1).astype(np.float32)
        
        # Normalisation des entrées (Standard Scaler)
        X_mean, X_std = X_np.mean(), X_np.std()
        X_norm = (X_np - X_mean) / (X_std + 1e-6)
        
        X_tensor = torch.from_numpy(X_norm)
        Y_tensor = torch.from_numpy(Y_np)
        
        # Initialisation du modèle pour ce pas de temps
        model = Net()
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)
        criterion = nn.MSELoss()
        
        # Boucle d'entraînement
        for epoch in range(epochs):
            optimizer.zero_grad()
            outputs = model(X_tensor)
            loss = criterion(outputs, Y_tensor)
            loss.backward()
            optimizer.step()
            
        # Prédiction de la valeur de continuation
        with torch.no_grad():
            continuation_value = model(X_tensor).numpy().flatten()
            
        exercise_value = payoff(S[itm_indices, t], K)
        exercise = exercise_value > continuation_value
        
        value_matrix_nn[itm_indices, t] = np.where(exercise, exercise_value, value_matrix_nn[itm_indices, t+1] * discount_factor)
        
    otm_indices = np.where(payoff(S[:, t], K) == 0)[0]
    value_matrix_nn[otm_indices, t] = value_matrix_nn[otm_indices, t + 1] * discount_factor
    
    if t % 10 == 0:
        print(f"Pas de temps {t}/{M} traité.")

ls_nn_price = np.mean(value_matrix_nn[:, 1] * discount_factor)
print(f"Prix Put Américain (Longstaff-Schwartz NN) : {ls_nn_price:.4f}")

import pandas as pd

results = {
    "Méthode": ["Black-Scholes (Européen)", "Longstaff-Schwartz (Polynomial)", "Longstaff-Schwartz (Neural Network)"],
    "Prix Estimé": [bs_price, ls_poly_price, ls_nn_price]
}

df_results = pd.DataFrame(results)
print(df_results)

print("\nConclusion:")
print("On observe que le prix de l'option américaine est supérieur au prix européen (prime d'exercice anticipé).")
print("La méthode par réseaux de neurones donne des résultats proches de la régression polynomiale classique.")